In [48]:
import pandas as pd
import numpy as np

In [49]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")

In [50]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

classes = [
    'nevus',
    'melanoma',
    'benign keratosis',
    'basal cell carcinoma',
    'actinic keratosis'
]

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    horizontal_flip=True,
    zoom_range=0.2
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [51]:
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='image_path',
    y_col='label',
    target_size=(160,160),
    batch_size=16,
    class_mode='categorical',
    #classes=classes
)

val_generator = val_datagen.flow_from_dataframe(
    val_df,
    x_col='image_path',
    y_col='label',
    target_size=(160,160),
    batch_size=16,
    class_mode='categorical',
    #classes=classes
)

Found 6830 validated image filenames belonging to 5 classes.
Found 1464 validated image filenames belonging to 5 classes.


In [52]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(5.965065502183406), 1: np.float64(3.7944444444444443), 2: np.float64(1.776332899869961), 3: np.float64(1.7535301668806162), 4: np.float64(0.29107180907734925)}


In [53]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(160,160,3)
)

In [54]:
# freeze most layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

# unfreeze last layers
for layer in base_model.layers[-100:]:
    layer.trainable = True

In [55]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(5, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [56]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [57]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

In [58]:
print("Train:", train_generator.class_indices)
print("Val:", val_generator.class_indices)

Train: {'actinic keratosis': 0, 'basal cell carcinoma': 1, 'benign keratosis': 2, 'melanoma': 3, 'nevus': 4}
Val: {'actinic keratosis': 0, 'basal cell carcinoma': 1, 'benign keratosis': 2, 'melanoma': 3, 'nevus': 4}


In [59]:
print(train_df['label'].value_counts())

label
nevus                   4693
melanoma                 779
benign keratosis         769
basal cell carcinoma     360
actinic keratosis        229
Name: count, dtype: int64


In [60]:
batch = next(train_generator)

print(batch[0].shape)  # images
print(batch[1].shape)  # labels

(16, 160, 160, 3)
(16, 5)


In [61]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    class_weight=class_weights,   # 🔥 MUST
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/8
427/427 ━━━━━━━━━━━━━━━━━━━━ 473s 1s/step - accuracy: 0.4332 - loss: 1.4397 - val_accuracy: 0.6503 - val_loss: 0.8981 - learning_rate: 5.0000e-05
Epoch 2/8
427/427 ━━━━━━━━━━━━━━━━━━━━ 488s 1s/step - accuracy: 0.6126 - loss: 1.0827 - val_accuracy: 0.6796 - val_loss: 0.7935 - learning_rate: 5.0000e-05
Epoch 3/8
427/427 ━━━━━━━━━━━━━━━━━━━━ 449s 1s/step - accuracy: 0.6493 - loss: 0.9403 - val_accuracy: 0.6749 - val_loss: 0.7686 - learning_rate: 5.0000e-05
Epoch 4/8
427/427 ━━━━━━━━━━━━━━━━━━━━ 439s 1s/step - accuracy: 0.6754 - loss: 0.8524 - val_accuracy: 0.6885 - val_loss: 0.7457 - learning_rate: 5.0000e-05
Epoch 5/8
427/427 ━━━━━━━━━━━━━━━━━━━━ 1263s 3s/step - accuracy: 0.6963 - loss: 0.7510 - val_accuracy: 0.6919 - val_loss: 0.7309 - learning_rate: 5.0000e-05
Epoch 6/8
427/427 ━━━━━━━━━━━━━━━━━━━━ 447s 1s/step - accuracy: 0.7085 - loss: 0.7187 - val_accuracy: 0.7370 - val_loss: 0.6412 - learning_rate: 5.0000e-05
Epoch 7/8
427/427 ━━━━━━━━━━━━━━━━━━━━ 474s 1s/step - accuracy:

In [62]:
model.save("../models/efficientnet_model_5class.h5")